# Human-in-the-Loop Review Interface

This notebook provides an interactive interface to review detection results and filter false positives before segmentation.

## Workflow
1. Load detection results
2. Visualize each detection with bounding boxes
3. Accept/Reject/Skip each image
4. Save accepted images for segmentation

In [ ]:
import os
import sys
import pickle
import shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import numpy as np

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))
from src.detection import CoolingTowerDetector
from src.visualization import draw_boxes_on_image

## Configuration

In [ ]:
# Paths
DETECTION_FILE = "../outputs/detections/detections.pkl"
REVIEW_DIR = "../outputs/review_images"  # Visualizations with boxes
ACCEPTED_DIR = "../outputs/accepted_grids"  # Accepted original images
REJECTED_DIR = "../outputs/rejected_grids"  # Rejected images

# Settings
CONF_THRESHOLD = 0.4  # Only review detections above this confidence

# Create directories
os.makedirs(REVIEW_DIR, exist_ok=True)
os.makedirs(ACCEPTED_DIR, exist_ok=True)
os.makedirs(REJECTED_DIR, exist_ok=True)

print("Configuration loaded!")
print(f"Detection file: {DETECTION_FILE}")
print(f"Review images will be saved to: {REVIEW_DIR}")

## Load Detection Results

In [ ]:
# Load detections
with open(DETECTION_FILE, 'rb') as f:
    all_detections = pickle.load(f)

print(f"Loaded {len(all_detections)} detection results")

# Filter for images with detections above threshold
detections_to_review = [
    d for d in all_detections 
    if any(c >= CONF_THRESHOLD for c in d['confidences'])
]

print(f"Found {len(detections_to_review)} images with detections above threshold")

## Generate Review Visualizations

In [ ]:
# Generate visualizations for review
print("Generating review visualizations...")

for detection in detections_to_review:
    img_path = detection['image_path']
    boxes = detection['boxes']
    confs = detection['confidences']
    
    filename = os.path.basename(img_path)
    output_path = os.path.join(REVIEW_DIR, filename)
    
    draw_boxes_on_image(
        img_path, boxes, confs, output_path,
        conf_threshold=CONF_THRESHOLD
    )

print("Visualizations complete!")

## Interactive Review Interface

In [ ]:
# Get list of images to review
review_images = sorted(os.listdir(REVIEW_DIR))
img_idx = 0
selected_images = []  # Store accepted images

# Output widget
out = widgets.Output()

# Navigation buttons
button_accept = widgets.Button(
    description="✔ Accept",
    button_style='success',
    tooltip='Accept this detection'
)
button_reject = widgets.Button(
    description="✘ Reject",
    button_style='danger',
    tooltip='Reject this detection'
)
button_skip = widgets.Button(
    description="➡ Skip",
    button_style='info',
    tooltip='Skip for now'
)

# Progress label
progress_label = widgets.Label(value="")

def update_progress():
    progress_label.value = f"Image {img_idx + 1}/{len(review_images)} | Accepted: {len(selected_images)}"

def handle_decision(decision):
    global img_idx
    
    if img_idx < len(review_images):
        img_filename = review_images[img_idx]
        review_path = os.path.join(REVIEW_DIR, img_filename)
        
        # Find original image path
        original_path = None
        for detection in detections_to_review:
            if os.path.basename(detection['image_path']) == img_filename:
                original_path = detection['image_path']
                break
        
        if decision == 'accept' and original_path:
            selected_images.append(img_filename)
            shutil.copy(original_path, os.path.join(ACCEPTED_DIR, img_filename))
            shutil.copy(review_path, os.path.join(ACCEPTED_DIR, f"review_{img_filename}"))
        
        elif decision == 'reject' and original_path:
            shutil.copy(original_path, os.path.join(REJECTED_DIR, img_filename))
        
        img_idx += 1
        display_image()

def display_image():
    with out:
        clear_output(wait=True)
        update_progress()
        
        if img_idx < len(review_images):
            img_path = os.path.join(REVIEW_DIR, review_images[img_idx])
            display(Image(filename=img_path, width=700))
        else:
            print("\n" + "="*60)
            print("✅ REVIEW COMPLETE!")
            print("="*60)
            print(f"Total images reviewed: {len(review_images)}")
            print(f"Accepted: {len(selected_images)}")
            print(f"Rejected: {len(review_images) - len(selected_images)}")
            print(f"\nAccepted images saved to: {ACCEPTED_DIR}")
            
            # Save selected list
            with open(os.path.join(ACCEPTED_DIR, 'selected_images.pkl'), 'wb') as f:
                pickle.dump(selected_images, f)
            print(f"Selected list saved to: {ACCEPTED_DIR}/selected_images.pkl")
            
            # Disable buttons
            button_accept.disabled = True
            button_reject.disabled = True
            button_skip.disabled = True

# Connect buttons
button_accept.on_click(lambda b: handle_decision('accept'))
button_reject.on_click(lambda b: handle_decision('reject'))
button_skip.on_click(lambda b: handle_decision('skip'))

# Layout
button_box = widgets.HBox([button_accept, button_reject, button_skip])
ui = widgets.VBox([progress_label, button_box, out])

# Display interface
display(ui)
display_image()

## Review Statistics

In [ ]:
# After review is complete, analyze results
print("Review Statistics:")
print("="*60)
print(f"Total images reviewed: {len(review_images)}")
print(f"Accepted: {len(selected_images)} ({len(selected_images)/len(review_images)*100:.1f}%)")
print(f"Rejected: {len(review_images) - len(selected_images)} ({(len(review_images)-len(selected_images))/len(review_images)*100:.1f}%)")
print("="*60)

## Next Steps

After review:
1. The accepted images are in `outputs/accepted_grids/`
2. Run segmentation only on accepted images:

```bash
python scripts/run_segmentation.py \
    --input_dir outputs/accepted_grids \
    --detection_file outputs/detections/detections.pkl \
    --output_dir outputs/masks
```